# Final Model - Depression Risk Classification

**Objective:** Build a machine learning model to classify depression risk into three categories: **Low, Medium, and High**.

**Data Integration:** After studying each dataset independently, we identified common and important features from both datasets. We will merge them into one unified dataset, prepare all necessary features, train our model, and analyze the results.

**Scope:** Data loading, Rescaling ,merging, EDA, feature preparation, model training, and depression risk classification.

# Introduction

Our approach combines data from two independent sources with the same important features. By merging these datasets and using machine learning, we can predict whether a student is at low, medium, or high risk of depression.

**Workflow:**
- **Load & Explore:** Import and understand both datasets
- **Handle Ranges (Rescaling)**
- **Merge Data:** Create one unified dataset
- **Train & Analyze:** Build prediction models and analyze results
- **Classify Risk:** Categorize students into risk levels

# Notebook Setup

In [5]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Section 1 : Data Loading & Initial Overview

### Objective
Load the datasets and obtain fundamental information about its structure, dimensions, and data types.


In [6]:
# Load the dataset
from pathlib import Path
data_path = Path('../data')

ds_a = pd.read_csv(data_path / 'ds_a.csv')
ds_b = pd.read_csv(data_path / 'ds_b.csv')

#### Dataset A

In [7]:
# Display basic information
print("="*80)
print("DATASET DIMENSIONS")
print("="*80)
print(f"Rows: {ds_a.shape[0]:,}")
print(f"Columns: {ds_a.shape[1]}")

print("First Few Records:")
print("=" * 80)
ds_a.head()

DATASET DIMENSIONS
Rows: 27,901
Columns: 5
First Few Records:


,CGPA,Sleep Duration,Work/Study Hours,Academic Pressure,Depression
0,8.97,5.5,3,5,1
1,5.90,5.5,3,2,0
2,7.03,4.5,9,3,0
3,5.59,7.5,4,3,1
4,8.13,5.5,1,4,0


In [9]:
# Display column names and target distribution
print("="*80)
print("COLUMN NAMES & DATA TYPES")
print("="*80)
print(ds_a.dtypes)

print("\n" + "=" * 80)
print("TARGET VARIABLE DISTRIBUTION:")
print("=" * 80)
print(ds_a['Depression'].value_counts())
print(f"\nClass Distribution: {ds_a['Depression'].value_counts(normalize=True).values}")
print(f"Depression Rate: {(ds_a['Depression'].sum() / len(ds_a) * 100):.2f}%")

COLUMN NAMES & DATA TYPES
CGPA                 float64
Sleep Duration       float64
Work/Study Hours       int64
Academic Pressure      int64
Depression             int64
dtype: object

TARGET VARIABLE DISTRIBUTION:
Depression
1    16336
0    11565
Name: count, dtype: int64

Class Distribution: [0.58549873 0.41450127]
Depression Rate: 58.55%


#### Dataset B

In [11]:
# Display basic information
print("="*80)
print("DATASET DIMENSIONS")
print("="*80)
print(f"Rows: {ds_b.shape[0]:,}")
print(f"Columns: {ds_b.shape[1]}")

print("First Few Records:")
print("=" * 80)
ds_b.head()

DATASET DIMENSIONS
Rows: 30,062
Columns: 5
First Few Records:


,CGPA,Sleep_Duration,Study_Hours,Stress_Level,Depression
0,2.55,7.2,7.0,3,0
1,2.28,8.2,1.2,5,0
2,3.42,6.9,1.9,2,0
3,2.93,6.6,5.9,2,0
4,3.30,6.2,6.4,3,0


In [12]:
# Display column names and target distribution
print("="*80)
print("COLUMN NAMES & DATA TYPES")
print("="*80)
print(ds_b.dtypes)

print("\n" + "=" * 80)
print("TARGET VARIABLE DISTRIBUTION:")
print("=" * 80)
print(ds_b['Depression'].value_counts())
print(f"\nClass Distribution: {ds_b['Depression'].value_counts(normalize=True).values}")
print(f"Depression Rate: {(ds_b['Depression'].sum() / len(ds_b) * 100):.2f}%")

COLUMN NAMES & DATA TYPES
CGPA              float64
Sleep_Duration    float64
Study_Hours       float64
Stress_Level        int64
Depression          int64
dtype: object

TARGET VARIABLE DISTRIBUTION:
Depression
0    20000
1    10062
Name: count, dtype: int64

Class Distribution: [0.66529173 0.33470827]
Depression Rate: 33.47%


# Section 2 : Data Profiling and Cleaning

### Objective
Understand the structure, completeness, and quality of each feature through basic statistical summaries.


#### Dataset A

In [13]:
# Summary statistics for numerical columns
print("\n" + "=" * 80)
print("NUMERICAL FEATURES - SUMMARY STATISTICS:")
print("=" * 80)
print(ds_a.describe().round(3))


NUMERICAL FEATURES - SUMMARY STATISTICS:
            CGPA  Sleep Duration  Work/Study Hours  Academic Pressure  \
count  27901.000       27901.000         27901.000          27901.000   
mean       7.657           6.379             7.157              3.141   
std        1.467           1.590             3.708              1.381   
min        2.345           4.500             0.000              0.000   
25%        6.290           4.500             4.000              2.000   
50%        7.770           5.500             8.000              3.000   
75%        8.920           7.500            10.000              4.000   
max       10.000           8.500            12.000              5.000   

       Depression  
count   27901.000  
mean        0.585  
std         0.493  
min         0.000  
25%         0.000  
50%         1.000  
75%         1.000  
max         1.000  


#### Dataset B

In [14]:
# Summary statistics for numerical columns
print("\n" + "=" * 80)
print("NUMERICAL FEATURES - SUMMARY STATISTICS:")
print("=" * 80)
print(ds_b.describe().round(3))


NUMERICAL FEATURES - SUMMARY STATISTICS:
            CGPA  Sleep_Duration  Study_Hours  Stress_Level  Depression
count  30062.000       30062.000     30062.00     30062.000   30062.000
mean       2.822           6.915         4.48         4.172       0.335
std        0.547           1.524         1.94         1.439       0.472
min        1.650           3.000         0.00         2.000       0.000
25%        2.350           5.900         3.10         3.000       0.000
50%        2.780           7.000         4.50         4.000       0.000
75%        3.290           8.000         5.80         5.000       1.000
max        4.000          11.100         9.80         8.000       1.000


#### Check Data Quality (Duplicated rows)

In [17]:
# DATA QUALITY CHECKS & DUPLICATE REMOVAL:
print("\n" + "=" * 80)
print("DATA QUALITY CHECKS - BEFORE CLEANING:")
print("=" * 80)
print(f"Dataset A - Duplicate rows: {ds_a.duplicated().sum()}")
print(f"Dataset A - Rows with all NaN: {ds_a.isnull().all(axis=1).sum()}")
print(f"Dataset A - Original shape: {ds_a.shape}")

print(f"\nDataset B - Duplicate rows: {ds_b.duplicated().sum()}")
print(f"Dataset B - Rows with all NaN: {ds_b.isnull().all(axis=1).sum()}")
print(f"Dataset B - Original shape: {ds_b.shape}")

# Remove duplicate rows, keeping the first occurrence
ds_a = ds_a.drop_duplicates(keep='first')
ds_b = ds_b.drop_duplicates(keep='first')

print("\n" + "=" * 80)
print("DATA QUALITY CHECKS - AFTER REMOVING DUPLICATES:")
print("=" * 80)
print(f"Dataset A - New shape: {ds_a.shape}")
print(f"Dataset B - New shape: {ds_b.shape}")
print(f"\nDataset A - Remaining duplicate rows: {ds_a.duplicated().sum()}")
print(f"Dataset B - Remaining duplicate rows: {ds_b.duplicated().sum()}")


DATA QUALITY CHECKS - BEFORE CLEANING:
Dataset A - Duplicate rows: 5493
Dataset A - Rows with all NaN: 0
Dataset A - Original shape: (27901, 5)

Dataset B - Duplicate rows: 102
Dataset B - Rows with all NaN: 0
Dataset B - Original shape: (30062, 5)

DATA QUALITY CHECKS - AFTER REMOVING DUPLICATES:
Dataset A - New shape: (22408, 5)
Dataset B - New shape: (29960, 5)

Dataset A - Remaining duplicate rows: 0
Dataset B - Remaining duplicate rows: 0


# Section 3 : Data Rescaling

**Validate value ranges** for each feature


In [ ]:
# Create a copy
ds_a_rescaled = ds_a.copy()
ds_b_rescaled = ds_b.copy()

print("Starting data cleaning process...")
print(f"Initial shape: {ds_a_rescaled.shape}")
print(f"Initial shape: {ds_b_rescaled.shape}")

Starting data cleaning process...
Initial shape: (22408, 5)
Initial shape: (29960, 5)


### Step 1: Rescale gpa data A to 0 - 4


In [21]:
ds_a_rescaled["CGPA"] = ds_a_rescaled["CGPA"] * (4 / 10)

### Step 2: Rescale sleep in data A (3 - 11)
Same distribution on the same scale

In [22]:
ds_a_rescaled["Sleep Duration"] = (
    (ds_a_rescaled["Sleep Duration"] - 4.5) / (8.5 - 4.5)
) * (11 - 3) + 3

### Step 3: Rescale stress in dataset B to 1 - 5


In [23]:
ds_b_rescaled["Stress_Level"] = (
    (ds_b_rescaled["Stress_Level"] - 2) / (8 - 2)
) * (5 - 0) + 0

### Step 4: Rename features

In [31]:
ds_a_rescaled.rename(columns={
    "Academic Pressure": "stress",
    "Sleep Duration": "sleep",
    "Work/Study Hours": "study_hours",
    "CGPA": "gpa"
}, inplace=True)

ds_b_rescaled.rename(columns={
    "Stress_Level": "stress",
    "Sleep_Duration": "sleep",
     "CGPA": "gpa",
     
}, inplace=True)

# Section 3.5 : Merge Datasets

### Objective
Combine Dataset A and Dataset B into one unified dataset for model training.

In [ ]:
# Merge datasets
df_a = ds_a_rescaled.copy()
df_b = ds_b_rescaled.copy()

# Add source column
df_a['source'] = 'Dataset_A'
df_b['source'] = 'Dataset_B'

# Concatenate both datasets
df_merged = pd.concat([df_a, df_b], axis=0, ignore_index=True)

print("="*80)
print("DATASETS MERGED SUCCESSFULLY")
print("="*80)
print(f"Dataset A rows: {len(df_a):,}")
print(f"Dataset B rows: {len(df_b):,}")
print(f"Merged dataset rows: {len(df_merged):,}")
print(f"\nMerged dataset shape: {df_merged.shape}")
print(f"\nColumn names: {list(df_merged.columns)}")
print(f"\nData types:\n{df_merged.dtypes}")
print(f"\nFirst few rows:")
print(df_merged.head())

# Section 4 : Exploratory Data Analysis (EDA)

### Objective
Analyze the merged dataset to understand feature distributions, correlations, and target variable balance.

In [ ]:
# Display merged dataset summary
print("="*80)
print("MERGED DATASET OVERVIEW")
print("="*80)
print(f"Shape: {df_merged.shape}")
print(f"\nBasic Statistics:")
print(df_merged.describe().round(3))
print(f"\nMissing Values:")
print(df_merged.isnull().sum())
print(f"\nSource Distribution:")
print(df_merged['source'].value_counts())

In [ ]:
# Analyze target variable
print("\n" + "="*80)
print("TARGET VARIABLE ANALYSIS")
print("="*80)
print(f"Depression Distribution:")
print(df_merged['Depression'].value_counts())
print(f"\nClass Balance:")
print(df_merged['Depression'].value_counts(normalize=True) * 100)
depression_rate = (df_merged['Depression'].sum() / len(df_merged)) * 100
print(f"\nDepression Rate: {depression_rate:.2f}%")

In [ ]:
# Feature correlations with target
features = ['gpa', 'sleep', 'study_hours', 'stress']
print("\n" + "="*80)
print("FEATURE-TARGET CORRELATIONS")
print("="*80)
correlations = df_merged[features + ['Depression']].corr()['Depression'].drop('Depression')
print(correlations.sort_values(ascending=False))

# Correlation heatmap
plt.figure(figsize=(8, 6))
corr_matrix = df_merged[features + ['Depression']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlations with Depression')
plt.tight_layout()
plt.show()

# Section 5: Prepare Data for Model Training

### Objective
Split data into training and testing sets. Scale features ONLY for Logistic Regression (not needed for Random Forest).

In [ ]:
# Import sklearn libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report, roc_curve)

# Select features and target
features = ['gpa', 'sleep', 'study_hours', 'stress']
X = df_merged[features]
y = df_merged['Depression']

print("="*80)
print("TRAIN-TEST SPLIT")
print("="*80)
print(f"Total samples: {len(X):,}")
print(f"Feature columns: {features}")

# Stratified train-test split (keep same depression ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")
print(f"\nTraining set depression rate: {(y_train.sum()/len(y_train))*100:.2f}%")
print(f"Test set depression rate: {(y_test.sum()/len(y_test))*100:.2f}%")

In [ ]:
# Scale features ONLY for Logistic Regression
# Random Forest doesn't need scaling (tree-based model)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=features, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=features, index=X_test.index)

print("\n" + "="*80)
print("FEATURE SCALING (for Logistic Regression only)")
print("="*80)
print(f"Scaler: StandardScaler")
print(f"\nScaled training data statistics:")
print(X_train_scaled.describe().round(3))
print(f"\nNote: Random Forest will use unscaled features (X_train, X_test)")

# Section 6: Model Training & Evaluation

### Objective
Train Random Forest and Logistic Regression models, then evaluate their performance.

### 6.2 Import ML Libraries

Import all necessary libraries for model training and evaluation

In [ ]:
# Train Random Forest Classifier
print("="*80)
print("TRAINING RANDOM FOREST CLASSIFIER")
print("="*80)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
print("✓ Random Forest trained successfully")
print(f"Number of trees: {rf_model.n_estimators}")
print(f"Max depth: {rf_model.max_depth}")

8.2 MACHINE LEARNING LIBRARIES IMPORTED

✓ Models: RandomForestClassifier, LogisticRegression
✓ Metrics: Accuracy, Precision, Recall, F1-Score, ROC-AUC
✓ Evaluation: Confusion Matrix, Classification Report, ROC Curve


### 6.3 Train Random Forest Classifier


In [ ]:
# Train Logistic Regression with scaled features
print("\n" + "="*80)
print("TRAINING LOGISTIC REGRESSION")
print("="*80)

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

lr_model.fit(X_train_scaled, y_train)
print("✓ Logistic Regression trained successfully (on scaled features)")
print(f"Solver: lbfgs (default)")
print(f"Max iterations: 1000")


8.4 HYPERPARAMETER TUNING - RANDOM FOREST

[1] Starting Grid Search (this may take a few minutes)...
Fitting 5 folds for each of 48 candidates, totalling 240 fits

[2] Best Parameters Found:
{'class_weight': 'balanced', 'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 10, 'n_estimators': 200}


### 6.4 Train Logistic Regression

**Why Logistic Regression?**
- Linear classification model (interpretable)
- Fast training and prediction
- Provides coefficients showing feature impact
- Good baseline for comparison
- Requires feature scaling for optimal performance

**Configuration**:
- `max_iter=1000`: Max iterations for convergence
- `random_state=42`: Reproducibility
- `solver='lbfgs'`: Good for small-medium datasets
- `class_weight='balanced'`: Handle class imbalance
- `C=1.0`: Regularization strength

# Section 6.1: Model Evaluation

### Metrics Explained
- **Accuracy**: Overall correctness (%)
- **Precision**: Of predicted depressed, how many actually are?
- **Recall**: Of actual depressed, how many were found?
- **F1-Score**: Balance between Precision and Recall
- **ROC-AUC**: Overall model discrimination ability

In [ ]:
# Make predictions
y_pred_rf = rf_model.predict(X_test)
y_pred_lr = lr_model.predict(X_test_scaled)

y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate Random Forest
print("="*80)
print("RANDOM FOREST - PERFORMANCE METRICS")
print("="*80)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_rf):.4f}")

# Evaluate Logistic Regression
print("\n" + "="*80)
print("LOGISTIC REGRESSION - PERFORMANCE METRICS")
print("="*80)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_lr):.4f}")

In [ ]:
# Confusion matrices
print("\n" + "="*80)
print("CONFUSION MATRICES")
print("="*80)

cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_lr = confusion_matrix(y_test, y_pred_lr)

print("\nRandom Forest:")
print(f"True Negatives: {cm_rf[0,0]:,} | False Positives: {cm_rf[0,1]:,}")
print(f"False Negatives: {cm_rf[1,0]:,} | True Positives: {cm_rf[1,1]:,}")

print("\nLogistic Regression:")
print(f"True Negatives: {cm_lr[0,0]:,} | False Positives: {cm_lr[0,1]:,}")
print(f"False Negatives: {cm_lr[1,0]:,} | True Positives: {cm_lr[1,1]:,}")

In [ ]:
# Feature Importance Analysis
print("\n" + "="*80)
print("FEATURE IMPORTANCE - RANDOM FOREST")
print("="*80)
feature_importance_rf = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance_rf.to_string(index=False))

print("\n" + "="*80)
print("FEATURE COEFFICIENTS - LOGISTIC REGRESSION")
print("="*80)
feature_coef_lr = pd.DataFrame({
    'feature': features,
    'coefficient': lr_model.coef_[0]
}).sort_values('coefficient', ascending=False)

print(feature_coef_lr.to_string(index=False))
print(f"\nIntercept: {lr_model.intercept_[0]:.4f}")
print("\nNote: Positive coefficient = increases depression risk")
print("      Negative coefficient = decreases depression risk")

### 6.6 Feature Importance Analysis

**Random Forest Importance**: Based on how much each feature splits the data (higher = more important)

**Logistic Regression Coefficients**:
- Positive coefficients → increases depression risk
- Negative coefficients → decreases depression risk
- Larger magnitude → stronger effect

# Section 7: Risk Profiling & Student Segmentation

### Objective
Classify students into risk groups based on depression prediction probability.

### Risk Categories
- **HIGH RISK**: Depression probability > 70%
- **MEDIUM RISK**: Depression probability 40-70%
- **LOW RISK**: Depression probability < 40%

In [ ]:
# Create risk groups using Random Forest (better performance)
# Use the most important model for risk classification
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

def classify_risk(prob):
    if prob > 0.70:
        return 'HIGH RISK'
    elif prob > 0.40:
        return 'MEDIUM RISK'
    else:
        return 'LOW RISK'

risk_groups = pd.Series(y_pred_proba).apply(classify_risk)

print("="*80)
print("RISK STRATIFICATION RESULTS (Based on Random Forest)")
print("="*80)
print(f"\nRisk Group Distribution:")
print(risk_groups.value_counts())
print(f"\nRisk Group Percentages:")
print(risk_groups.value_counts(normalize=True) * 100)

# Analyze each risk group
print("\n" + "="*80)
print("DEPRESSION RATE BY RISK GROUP")
print("="*80)

risk_df = pd.DataFrame({
    'risk_group': risk_groups,
    'actual_depression': y_test.values,
    'predicted_probability': y_pred_proba
})

for risk in ['HIGH RISK', 'MEDIUM RISK', 'LOW RISK']:
    group = risk_df[risk_df['risk_group'] == risk]
    depression_rate = (group['actual_depression'].sum() / len(group)) * 100
    print(f"\n{risk}:")
    print(f"  Students: {len(group):,}")
    print(f"  Actual depression rate: {depression_rate:.2f}%")
    print(f"  Mean predicted probability: {group['predicted_probability'].mean():.4f}")

In [ ]:
# Visualize risk distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Risk group distribution
risk_counts = risk_groups.value_counts()
axes[0].bar(risk_counts.index, risk_counts.values, color=['red', 'orange', 'green'])
axes[0].set_title('Distribution of Students by Risk Group')
axes[0].set_ylabel('Number of Students')
axes[0].grid(axis='y', alpha=0.3)

# Model performance comparison
models = ['Random Forest', 'Logistic Regression']
accuracies = [
    accuracy_score(y_test, y_pred_rf),
    accuracy_score(y_test, y_pred_lr)
]
roc_aucs = [
    roc_auc_score(y_test, y_pred_proba_rf),
    roc_auc_score(y_test, y_pred_proba_lr)
]

x = np.arange(len(models))
width = 0.35
axes[1].bar(x - width/2, accuracies, width, label='Accuracy', alpha=0.8)
axes[1].bar(x + width/2, roc_aucs, width, label='ROC-AUC', alpha=0.8)
axes[1].set_ylabel('Score')
axes[1].set_title('Model Performance Comparison')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models)
axes[1].legend()
axes[1].set_ylim([0, 1])
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Section 8: Summary & Conclusions

In [ ]:
print("="*80)
print("FINAL MODEL SUMMARY")
print("="*80)

print(f"\n📊 DATASETS:")
print(f"  • Dataset A: {len(df_a):,} students")
print(f"  • Dataset B: {len(df_b):,} students")
print(f"  • Total: {len(df_merged):,} students")

print(f"\n🎯 TARGET VARIABLE:")
print(f"  • Depression Rate: {(df_merged['Depression'].sum() / len(df_merged))*100:.2f}%")
print(f"  • Depressed Students: {df_merged['Depression'].sum():,}")
print(f"  • Non-Depressed Students: {(df_merged['Depression'] == 0).sum():,}")

print(f"\n📈 MODEL PERFORMANCE:")
print(f"  • Best Model: Random Forest")
print(f"  • Test Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"  • Test ROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.4f}")
print(f"  • Test Recall: {recall_score(y_test, y_pred_rf):.4f}")

print(f"\n📌 KEY FEATURES (by importance):")
for idx, row in feature_importance_rf.head(3).iterrows():
    print(f"  • {row['feature']}: {row['importance']:.4f}")

print(f"\n👥 RISK GROUPS:")
for risk in ['HIGH RISK', 'MEDIUM RISK', 'LOW RISK']:
    group = risk_df[risk_df['risk_group'] == risk]
    print(f"  • {risk}: {len(group):,} students ({len(group)/len(risk_df)*100:.1f}%)")

print(f"\n✓ Model training completed successfully!")
print(f"✓ Risk stratification complete!")
print("="*80)